TODO:
- Response ID tracking
- Perform edits with structured output
- Validated list tracking
- Switch from error to limitations
- Redefine sentence extraction

Generation:
- Change to storing results, not pydantic objects
- Prevent data loss in dill storage

In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import itertools
import config
import shared.taxonomy
import datetime
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema
from shared.utils.DataHandlers import LatexDataHandler
from shared.utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pylatexenc.latexwalker import LatexWalker, LatexMacroNode, LatexGroupNode
from pylatexenc.latex2text import LatexNodes2Text
from IPython.display import Image


In [3]:
data_handler = LatexDataHandler(config.DATA_FOLDER_PATH)

# Setup

In [4]:
OUTPUT_FOLDER_PATH = config.OUTPUT_FOLDER_PATH
WORKSPACE_FOLDER_PATH = config.WORKSPACE_FOLDER_PATH
RESULTS_FOLDER_PATH = config.RESULTS_FOLDER_PATH
EDIT_MODEL = config.EDIT_MODEL_INFO[2]

instance_id = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
BACKUP_FOLDER_PATH = f'backups/'
if not os.path.exists(BACKUP_FOLDER_PATH):
    os.makedirs(BACKUP_FOLDER_PATH)

if os.path.exists(OUTPUT_FOLDER_PATH) or os.path.exists(WORKSPACE_FOLDER_PATH) or os.path.exists(RESULTS_FOLDER_PATH):
    BACKUP_FOLDER_PATH = f'backups/backup_{instance_id}/'
    os.makedirs(BACKUP_FOLDER_PATH)

    if os.path.exists(OUTPUT_FOLDER_PATH):
        shutil.move(OUTPUT_FOLDER_PATH, BACKUP_FOLDER_PATH + OUTPUT_FOLDER_PATH)
    if os.path.exists(WORKSPACE_FOLDER_PATH):
        shutil.move(WORKSPACE_FOLDER_PATH, BACKUP_FOLDER_PATH + WORKSPACE_FOLDER_PATH)
    if os.path.exists(RESULTS_FOLDER_PATH):
        shutil.move(RESULTS_FOLDER_PATH, BACKUP_FOLDER_PATH + RESULTS_FOLDER_PATH)

# Dataset Construction

In [25]:
#subprocess.run('python generate_edits.py')
subprocess.run(['conda', 'run', '-n', 'general', 'python', 'generate_edits.py'])

Starting generate edits...
Starting queries...
Loading from cache...
Loading cached responses from output/responses_edit_latex_opus_5_cache.dill
[2503.15484___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0: empty response in cache, skipping load
[2503.15484___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0: empty response in cache, skipping load
[2503.15484___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0: empty response in cache, skipping load
[2503.15484___bG9naWNhbGx5X2ludmFsaWRfaW50ZXJwcmV0YXRpb24=: empty response in cache, skipping load
[2503.15484___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=: empty response in cache, skipping load
[2503.15484___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==: empty response in cache, skipping load
[2503.15484___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==: empty response in cache, skipping load
2502.10118
2501.09751
2509.01093
2508.16347
2507.13255
2412.17534
2503.04990
2509.06105
2407.11843
2503.24354
2502.11740
2508.16861
2508.19024
2504.17704
2502.14409
2510.27672
2503.15484
2510.06

/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class '__main__.Edit'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class '__main__.Edit'>: __main__.Edit has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot locate reference to <class '__main__.Flaw'>.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/general/lib/python3.13/site-packages/dill/_dill.py:422: PicklingWarning: Cannot pickle <class '__main__.Flaw'>: __main__.Flaw has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/Users/aadi.joshi/miniconda3/envs/ge

CompletedProcess(args=['conda', 'run', '-n', 'general', 'python', 'generate_edits.py'], returncode=0)

In [18]:
class Edit(BaseModel):
    text_span_to_replace: str
    new_text_span: str

class Flaw(BaseModel):
    refuse: bool
    reasoning: str
    sentences: list[str]
    edits: list[Edit]

In [26]:
def load_from_cache(filename):
    print(f"Loading cached responses from {filename}")
    with open(filename, 'rb') as f:
        while True:
            try:
                # Load the next object in the stream
                obj = dill.load(f)
                yield obj
            except EOFError:
                # End of file reached
                break
            # uncomment this for UnpicklingError: but load all responses and save a new cache file, otherwise future responses won't be saved properly. fix_cache() above has the code to do this.
            # except:
            #     pass

responses = list(load_from_cache('./output/responses_edit_latex_opus_5_cache.dill'))

Loading cached responses from ./output/responses_edit_latex_opus_5_cache.dill


In [79]:
#subprocess.run('python perform_edits.py')
subprocess.run(['conda', 'run', '-n', 'general', 'python', 'validate_responses.py'])

CompletedProcess(args=['conda', 'run', '-n', 'general', 'python', 'validate_responses.py'], returncode=0)

In [83]:
with open(f'output/responses_edit_latex_{EDIT_MODEL}_validated.dill', 'rb') as fp:
    validated = dill.load(fp)

In [84]:
sum([int(res[2]['valid']) for res in validated])

264

In [82]:
#subprocess.run('python perform_edits.py')
subprocess.run(['conda', 'run', '-n', 'general', 'python', 'perform_edits.py'])

'utf-8' codec can't decode byte 0xf6 in position 31224: invalid start byte
'utf-8' codec can't decode byte 0xf6 in position 31396: invalid start byte
'utf-8' codec can't decode byte 0xf6 in position 31148: invalid start byte
'utf-8' codec can't decode byte 0xf6 in position 31986: invalid start byte
'utf-8' codec can't decode byte 0xe3 in position 60529: invalid continuation byte
'utf-8' codec can't decode byte 0xe3 in position 60777: invalid continuation byte
'utf-8' codec can't decode byte 0xe3 in position 60529: invalid continuation byte
'utf-8' codec can't decode byte 0xee in position 41317: invalid continuation byte
'utf-8' codec can't decode byte 0xee in position 41565: invalid continuation byte
'utf-8' codec can't decode byte 0xee in position 41317: invalid continuation byte
'utf-8' codec can't decode byte 0xee in position 41645: invalid continuation byte
'utf-8' codec can't decode byte 0xee in position 42155: invalid continuation byte
'utf-8' codec can't decode byte 0xe9 in posi

CompletedProcess(args=['conda', 'run', '-n', 'general', 'python', 'perform_edits.py'], returncode=0)

In [85]:
with open(f'output/responses_edit_latex_{EDIT_MODEL}_edited.dill', 'rb') as fp:
    edited = dill.load(fp)

In [86]:
with open(f'output/responses_edit_latex_{EDIT_MODEL}_compiled.dill', 'rb') as fp:
    compiled = dill.load(fp)

In [87]:
compiled

[('2502.10118',
  'bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0',
  {'sentences': ['The dataset annotations and features necessary to train ModeCap are exclusively available for the MSCOCO dataset, making it difficult to replicate the experiments for fair comparison on the NoCaps and Flickr30k datasets.'],
   'edit_list': [{'text_span_to_replace': 'ModeCap & \\xmark & - & - &\\cellcolor[HTML]{FFCCCC} 29.35 & \\cellcolor[HTML]{ADD8E6} 0.714 & \\cellcolor[HTML]{FFCCCC} 0.281 & \\cellcolor[HTML]{ADD8E6} 0.594 & - & - & - & - & -\\\\',
     'new_text_span': 'ModeCap & \\xmark & - & - &\\cellcolor[HTML]{FFCCCC} 29.35 & \\cellcolor[HTML]{ADD8E6} 0.714 & \\cellcolor[HTML]{FFCCCC} 0.281 & \\cellcolor[HTML]{ADD8E6} 0.594 & - & \\cellcolor[HTML]{FFCCCC} 28.90 & \\cellcolor[HTML]{ADD8E6} 0.705 & \\cellcolor[HTML]{FFCCCC} 0.290 & \\cellcolor[HTML]{ADD8E6} 0.580\\\\'},
    {'text_span_to_replace': '\\multicolumn{3}{c|}{HBoP \\textit{Ranking}} & 4/8 & 8/11 & 1/7 & 5/12 & 5/12 & 4/8 & 6/10 & 1/6 & 4/10 & 5/1

# Human Annotations

- Prepare data for upload
- Format and prepare downloaded annotations

In [ ]:
version_num = 1 + len([d for d in os.listdir("/Users/aadi.joshi/Desktop/scientific_fact_checking/tabular_scientific_fact_checking/code/manual_review/guided_error_introduction_edit_generation/") if d[0] == 'v'])
target_path = Path(f"/Users/aadi.joshi/Desktop/scientific_fact_checking/tabular_scientific_fact_checking/code/manual_review/guided_error_introduction_edit_generation/v{version_num}/")
if not target_path.exists():
    os.makedirs(target_path)

In [ ]:
compile_path = "/Users/aadi.joshi/Desktop/scientific_fact_checking/tabular_scientific_fact_checking/code/manual_review/guided_error_introduction_edit_generation/consolidate.py"
shutil.copy('./metadata.dill',target_path / 'metadata.dill')
shutil.copy('./utils/DataHandlers.py', target_path / 'DataHandlers.py')
shutil.copytree('./workspace', target_path / 'workspace', dirs_exist_ok=True)
shutil.copytree('./output', target_path / 'output', dirs_exist_ok=True)
shutil.copy(compile_path, target_path / 'consolidate.py')
subprocess.run(['conda', 'run', '-n', 'general', 'python', target_path / 'consolidate.py'], cwd=target_path)


In [ ]:
ANNOTATIONS_FILE_PATH = Path("/Users/aadi.joshi/Desktop/scientific_fact_checking/tabular_scientific_fact_checking/annotations/pilot_3/results")
annotation_results = []
for annotation_file in os.listdir(ANNOTATIONS_FILE_PATH):
    if annotation_file.endswith('.json'):
        with open(ANNOTATIONS_FILE_PATH / annotation_file, 'r') as fp:
            annotation_results.append(json.load(fp))

In [ ]:
for i, annotation_result in enumerate(annotation_results):
    for annotation in annotation_result:
        annotation['annotator_id'] = i

In [ ]:
results = {}

for annotator_result in annotation_results:
    for result in annotator_result:
        paper_id = result['data']['id']
        error_id = result['data']['error_id']
        meta_id = f'{paper_id}___{error_id}'

        #are these in the same order always? 
        if meta_id not in results:
            results[meta_id] = {}

        annotator = result['annotator_id']
        for annotation in result['annotations']:
            choice = None
            for res in annotation['result']:
                if res['type'] == 'choices' and res['from_name'] == 'edit_correct' and res['to_name'] == 'edited_pdf':
                    choice = res['value']['choices'][0]

            results[meta_id][annotator] = choice

In [ ]:
results

In [ ]:
NUM_ANNOTATORS = 2
for paper_id, error_id, compiled_res in compiled:
    meta_id = f'{paper_id}___{error_id}'

    if meta_id not in results:
        compiled_res['human_validation_passed'] = None

    if meta_id in results:
        annotations = results[meta_id]
        compiled_res['annotations'] = annotations
        num_yes = 0
        for annotator, choice in annotations.items():
            if choice == 'yes':
                num_yes += 1

        if num_yes == NUM_ANNOTATORS:
            compiled_res['human_validation_passed'] = True
        else:
            compiled_res['human_validation_passed'] = False

with open(f'output/annotations.dill', 'wb') as fp:
    dill.dump(compiled, fp)

In [ ]:
with open(f'output/annotations.dill', 'rb') as fp:
    ann = dill.load(fp)